In [1]:
# Download dataset (UHDM) and save 500 images
from pathlib import Path
import zipfile
from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

load_dotenv(Path(".env"))

api = KaggleApi()
api.authenticate()

dataset = "soumikrakshit/uhdm-dataset"
background_dir = Path("assets/recapture-check")
background_dir.mkdir(parents=True, exist_ok=True)

tmp_dir = Path("assets/_kaggle_tmp_uhdm")
tmp_dir.mkdir(parents=True, exist_ok=True)
api.dataset_download_files(dataset, path=str(tmp_dir), unzip=False, quiet=False, force=False)

zip_candidates = sorted(tmp_dir.glob("*.zip"))
if not zip_candidates:
  raise FileNotFoundError("dataset zip not found after download")
zip_path = zip_candidates[0]

if not zipfile.is_zipfile(zip_path):
  api.dataset_download_files(dataset, path=str(tmp_dir), unzip=False, quiet=False, force=True)
  if not zipfile.is_zipfile(zip_path):
    raise RuntimeError("dataset zip is incomplete; wait for download to finish and re-run")

image_exts = {".jpg", ".jpeg", ".png"}

with zipfile.ZipFile(zip_path, "r") as archive:
  members = [name for name in archive.namelist() if Path(name).suffix.lower() in image_exts]
  if not members:
    raise FileNotFoundError("no image files found in dataset zip")
  members = sorted(members)[:500]
  for name in members:
    target_path = background_dir / Path(name).name
    if target_path.exists():
      continue
    with archive.open(name) as source, open(target_path, "wb") as dest:
      dest.write(source.read())

Dataset URL: https://www.kaggle.com/datasets/soumikrakshit/uhdm-dataset
uhdm-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


KeyboardInterrupt: 

In [ ]:
# Train test split
from pathlib import Path
import random
import shutil

assets_dir = Path("assets/recapture-check")
train_pos_dir = Path("moire-pattern-detector/positiveImages")
train_neg_dir = Path("moire-pattern-detector/negativeImages")
test_pos_dir = assets_dir / "test" / "positive"
test_neg_dir = assets_dir / "test" / "negative"

for path in [train_pos_dir, train_neg_dir, test_pos_dir, test_neg_dir]:
  path.mkdir(parents=True, exist_ok=True)

image_paths = [
  path for path in assets_dir.iterdir()
  if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

random.shuffle(image_paths)
split_idx = int(len(image_paths) * 0.8)
train_paths = image_paths[:split_idx]
test_paths = image_paths[split_idx:]

for path in train_paths:
  name = path.name.lower()
  if "gt" in name:
    target_dir = train_neg_dir
  elif "moire" in name:
    target_dir = train_pos_dir
  else:
    continue
  target_path = target_dir / path.name
  if not target_path.exists():
    shutil.copy2(path, target_path)

for path in test_paths:
  name = path.name.lower()
  if "gt" in name:
    target_dir = test_neg_dir
  elif "moire" in name:
    target_dir = test_pos_dir
  else:
    continue
  target_path = target_dir / path.name
  if not target_path.exists():
    shutil.copy2(path, target_path)

In [4]:
# Run the recapture check
from pathlib import Path
import csv

from recapture_check import check_recapture

assets_dir = Path("assets/recapture-check")
labels_path = assets_dir / "recapture_check.csv"

labels = {}
with labels_path.open(newline="") as handle:
  reader = csv.reader(handle)
  for row in reader:
    if not row:
      continue
    name = row[0].strip()
    label = row[1].strip().lower() if len(row) > 1 else ""
    labels[name] = label

image_paths = sorted([path for path in assets_dir.iterdir() if path.suffix.lower() == ".jpg"])

correct = 0
wrong = 0
missing = 0

for path in image_paths:
  result = check_recapture(path.read_bytes())
  predicted_moire = not result.get("accept", False)
  label = labels.get(path.name)
  if label is None:
    missing += 1
    outcome = "no_label"
  else:
    expected_moire = label == "yes"
    outcome = "ok" if predicted_moire == expected_moire else "mismatch"
    if outcome == "ok":
      correct += 1
    else:
      wrong += 1
  print(f"{path.name} predicted_moire={predicted_moire} label={label} {outcome}")

print(f"correct={correct} wrong={wrong} missing_label={missing}")

IMG_6433.jpg predicted_moire=True label=yes ok
IMG_6434.jpg predicted_moire=False label=no ok
IMG_6435.jpg predicted_moire=False label=no ok
IMG_6436.jpg predicted_moire=True label=no mismatch
IMG_6437.jpg predicted_moire=True label=no mismatch
IMG_6438.jpg predicted_moire=False label=yes mismatch
IMG_6439.jpg predicted_moire=False label=yes mismatch
IMG_6440.jpg predicted_moire=True label=yes ok
correct=4 wrong=4 missing_label=0
